In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110
print("project root:", _root)

project root: /workspaces/geo_distance_arena


# 06 - Decision Framework

When to use which distance, with the evidence from the previous notebooks pulled together. This notebook reads the JSON each earlier notebook saved and assembles the headline numbers and the guidance.

In [2]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

from src.io_utils import DATA_DIR, save_fig, save_json

def load(name):
    return json.loads((DATA_DIR / f'{name}.json').read_text())

detour = load('03_detour_summary')
flips = load('04_flip_rate')
catch = load('04_catchment_counts')
hybrid = load('05_hybrid')
mode = load('02_mode_distances')

## The headline numbers

In [3]:
summary = {
    'detour_factor_mean': detour['mean'],
    'detour_factor_p90': detour['p90'],
    'detour_factor_max': detour['max'],
    'nearest_branch_flip_rate': flips['flip_rate'],
    'customers_inside_circle': catch['inside_circle'],
    'customers_inside_drive_time': catch['inside_drive_time'],
    'hybrid_agreement_rate': hybrid['agreement_rate'],
    'hybrid_speedup': hybrid['road_calls_full'] / hybrid['road_calls_hybrid'],
}
save_json(summary, '06_summary')
summary

{'detour_factor_mean': 1.3671219100864807,
 'detour_factor_p90': 1.6123412542599276,
 'detour_factor_max': 3.373983911895483,
 'nearest_branch_flip_rate': 0.10833333333333334,
 'customers_inside_circle': 32,
 'customers_inside_drive_time': 57,
 'hybrid_agreement_rate': 1.0,
 'hybrid_speedup': 8.0}

In [4]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['detour mean', 'detour p90', 'detour max', 'flip rate']
values = [summary['detour_factor_mean'], summary['detour_factor_p90'],
          summary['detour_factor_max'], summary['nearest_branch_flip_rate']]
colors = ['#4C78A8', '#4C78A8', '#4C78A8', '#E45756']
bars = ax.bar(labels, values, color=colors)
ax.axhline(1.0, color='#333', ls=':', lw=0.8)
ax.set_title('What the straight line misses (Copenhagen)')
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.2f}', ha='center', va='bottom')
fig.tight_layout()
save_fig(fig, '06_headline')
plt.show()

## The framework

| Use case | Metric | Why |
|---|---|---|
| Rough catchment, prefilter, huge datasets, dense grid city | **Straight-line** (geodesic for accuracy, haversine vectorised) | Instant; error is small and known. Never Euclidean on raw degrees. |
| Delivery cost, service-level radii, nearest-facility assignment, anything near a barrier | **Road distance** (OSMnx + NetworkX) | Straight-line is biased ~1/3 low on average and by multiples at barriers; a flat multiplier cannot fix it. |
| When the business cares how long it takes | **Travel time** (speed-weighted graph) | Free and much closer than distance; reach for a routing engine only for live traffic. |
| Many-to-many at scale | **Hybrid** (straight-line prefilter -> road rank) | Keeps the speed of straight-line and the truth of road where it matters. |

Feed the real distance (and time) matrix into the models that assume it: facility location in `supply_chain_arena` and routing in `routing_arena` are only as good as the distances you give them.

**Everything here is free, keyless, Docker-free, and offline after a one-time download.**